# Function Calling and Tool Use

> The previous sections covered how to train a model: how training loss is computed, how data is processed, how parameters are updated efficiently. Once trained, the model can already generate fluent text and answer common-knowledge questions.
>
> But there is a class of questions it cannot answer — today's weather, 357 times 289, the current commit count of a particular repository. All of these require external information or external computation. This section covers Function Call: through special data construction and training, the model learns to output structured call requests, which an external program executes and feeds the results back to the model.


LLM training data has a temporal cutoff, after which events are unknown to the model. LLMs are also not good at precise numerical computation — asking the model to directly compute 357 times 289 may produce a result that looks reasonable but is in fact wrong. These two limitations combined mean that pure text generation is insufficient in many practical scenarios.

The Function Call approach adds an external interface to the model. The model itself performs no operation; it only generates a structured piece of JSON specifying which function to call and what arguments to pass. This JSON is parsed and executed by an external program (for example, actually querying a weather API or invoking a calculator), and the execution result is then appended back into the conversation history. The model produces its final natural-language reply based on that result.

This capability is acquired through dedicated training; the model is not born knowing how to output JSON. Starting from the problem, the following sections walk through the complete Function Call flow and then look at how to train this capability.


## Section Highlights

After studying this section, you should be able to answer the following questions:

1. In which scenarios does an LLM need external tools?
2. What is the complete Function Call flow?
3. When a conversation requires multiple tool calls, how are parallel and sequential calls handled?
4. How do you train a model that supports Function Call?
5. What are the common failure modes of Function Call?
6. From Function Call to MCP, what is the evolution path of the tool-calling ecosystem?


## 1. Why Function Call Is Needed

An LLM's knowledge comes from training data, and training data has a temporal cutoff. Asked "what is the temperature in Beijing today", the model gives a vague answer based on weather statistics seen during training, but it cannot possibly know the real weather today. This is the first limitation — knowledge cutoff.

The second limitation is numerical computation. The fundamental operations of a Transformer are matrix multiplication and softmax, which are not suited to precise multi-digit multiplication. Asking the model to directly compute 357 × 289 often produces a wrong answer. The model's answer looks reasonable (the order of magnitude is right, the last digit may be right), but the middle digits are often incorrect.

The code below simulates these two failures.


In [ ]:
# Simulate two typical LLM failures: knowledge cutoff + numerical computation errors

# Scenario 1: knowledge cutoff
# The model's training data is in the past, so it cannot know "today's" real data
def fake_llm_today_weather():
    # An LLM that does not understand tools is asked about today's weather
    return "I cannot query Beijing's weather in real time, but the temperature this season is usually between 20-30C."

print("=== Failure 1: knowledge cutoff ===")
print("User: What is the temperature in Beijing today?")
print(f"Model: {fake_llm_today_weather()}")
print("Problem: the model does not know today's real weather and can only give a vague guess")
print()

# Scenario 2: numerical computation error
# Simulate a "model" that gets arithmetic wrong — it outputs an answer that looks reasonable but is wrong
def fake_llm_multiply(a, b):
    # Simulate an LLM computing multiplication; it may get it wrong
    correct = a * b
    wrong = correct + 1234  # deliberately give a wrong answer
    return wrong

a, b = 357, 289
correct = a * b
llm_answer = fake_llm_multiply(a, b)
print("=== Failure 2: numerical computation error ===")
print(f"User: what is {a} x {b}?")
print(f"Model: {llm_answer}")
print(f"Correct: {correct}")
print(f"Gap: the model's answer is {llm_answer - correct} higher than the correct answer")
print()
print("Conclusion: the model alone cannot solve these two kinds of problems. It needs an external interface.")


## 2. The Complete Function Call Flow

The core idea of Function Call is to separate "execution" from "decision":

- Decision (whether to call a tool, which one, what arguments to pass) is done by the model
- Execution (actually querying the weather, doing arithmetic, querying a database) is done by an external program

The complete flow has six steps. Let us walk through a concrete example: the user asks "What is the temperature in Beijing today?".

Step 1: define the tools. Each tool is a JSON description telling the model what the tool is called, what it does, and what parameters it needs.
Step 2: inject the tool description into the system prompt. The model sees this description before generating its reply.
Step 3: the model generates a reply. If it decides a tool is needed, instead of natural language it outputs a structured piece of JSON.
Step 4: the external program parses this JSON and calls the corresponding function.
Step 5: the execution result is wrapped as a new conversation message and added to the conversation history.
Step 6: the model generates the final natural-language reply based on the complete conversation history (including the tool result).


In [ ]:
# Step 1: define the available tools
# Each tool is a dict with three parts: name / description / parameters

import json

tools = [
    {
        "name": "get_weather",
        "description": "Query the current weather for a given city",
        "parameters": {
            "city": {"type": "string", "description": "City name"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit"}
        }
    },
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression",
        "parameters": {
            "expression": {"type": "string", "description": "Mathematical expression"}
        }
    }
]

print("=== Defined tools ===")
for t in tools:
    print(f"\nTool name: {t['name']}")
    print(f"  Description: {t['description']}")
    print(f"  Parameters:")
    for pname, pinfo in t['parameters'].items():
        print(f"    {pname} ({pinfo['type']}): {pinfo['description']}")


In [ ]:
# Step 2: assemble the tool descriptions into the system prompt
# This text is visible to the model, which uses it to decide whether and which tool to call

def build_system_prompt(tools):
    # Build a system prompt from the tool list
    lines = ["You are an assistant that can use the following tools. When external information is needed, output a call request in JSON format.", ""]
    lines.append("Available tools:")
    for t in tools:
        lines.append(f"\nTool: {t['name']}")
        lines.append(f"  Description: {t['description']}")
        lines.append(f"  Parameters:")
        for pname, pinfo in t['parameters'].items():
            lines.append(f"    - {pname}: {pinfo['description']}")
    return "\n".join(lines)

system_prompt = build_system_prompt(tools)
print("=== System Prompt (the model sees this) ===")
print(system_prompt)


In [ ]:
# Step 3: simulate the model generating a JSON call request
# In practice this is the token sequence produced by the model's forward pass; here we hand-simulate a reasonable output

user_query = "What is the temperature in Beijing today?"

# Suppose that, after training, the model faced with this user query and the tool list above outputs this JSON
model_output = json.dumps({
    "tool_call": {
        "name": "get_weather",
        "arguments": {"city": "Beijing", "unit": "celsius"}
    }
}, ensure_ascii=False, indent=2)

print("=== Model output ===")
print(f"User: {user_query}")
print("Instead of natural language, the model outputs JSON:")
print(model_output)
print()
print("Key observation: the model decides that external information is needed and selects the get_weather tool, filling in city='Beijing'.")


In [ ]:
# Step 4 + Step 5: an external program executes the tool, and the result is appended to the conversation history

# Simulate the real implementation of the tools
def execute_tool(name, arguments):
    # Execute the corresponding function based on the tool name
    if name == "get_weather":
        # In practice this would call a real weather API
        return {"temperature": 22, "condition": "sunny", "humidity": 45}
    elif name == "calculate":
        # In practice this would call a real calculator
        return {"result": eval(arguments["expression"])}
    return {"error": f"Unknown tool: {name}"}

# Parse the model's JSON output and execute the corresponding tool
parsed = json.loads(model_output)
call = parsed["tool_call"]
tool_result = execute_tool(call["name"], call["arguments"])

print("=== Tool execution ===")
print(f"Call: {call['name']}({call['arguments']})")
print(f"Return: {tool_result}")
print()

# Build the full conversation history, adding the tool result as a new message
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
    {"role": "assistant", "content": model_output},
    {"role": "tool", "name": call["name"], "content": json.dumps(tool_result, ensure_ascii=False)},
]

print("=== Full conversation history (now includes the tool result) ===")
for m in messages:
    role = m["role"]
    content = m["content"]
    if len(content) > 80:
        content = content[:80] + "..."
    print(f"[{role}] {content}")


In [ ]:
# Step 6: the model generates the final natural-language reply based on the full conversation history

# Simulate the model's final reply
final_reply = "Beijing is 22C and sunny today."

print("=== Final reply ===")
print(f"Model: {final_reply}")
print()
print("Key observation: after seeing the tool return temperature=22 and weather=sunny, the model composes them into a natural-language sentence.")
print("What the user perceives is an ordinary answer; the details of the tool call are hidden.")


## 3. Multi-Tool Scenarios

In real scenarios, a single question often needs multiple tools. For example, the user asks "What is the temperature in Beijing today? Also, what is 357 x 289?", and the model needs to call both get_weather and calculate.

There are two cases here, handled differently.

Parallel calls: the two tools have no dependency between them, so order does not affect the result. The model can generate multiple call requests in a single output, the external program executes them concurrently, and all results are fed back at once. This has lower latency.

Sequential calls: the arguments of the second tool depend on the result of the first. The model must call the first tool, see the result, and only then decide whether to call the second and what arguments to pass. For example, "check the temperature in Beijing today, then tell me what to wear" — the second call (clothing recommendation) depends on the result of the first call (weather). This case must proceed over multiple rounds.


In [ ]:
# Parallel calls: generate multiple tool call requests at once

user_query = "What is the temperature in Beijing today? Also, what is 357 x 289?"

# The model outputs multiple calls in one go
model_output = json.dumps({
    "tool_calls": [
        {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}},
        {"name": "calculate", "arguments": {"expression": "357*289"}}
    ]
}, ensure_ascii=False, indent=2)

print("=== Parallel call ===")
print(f"User: {user_query}")
print("Model output:")
print(model_output)
print()

# The external program parses and executes them one by one (in practice they can run concurrently)
parsed = json.loads(model_output)
for call in parsed["tool_calls"]:
    result = execute_tool(call["name"], call["arguments"])
    print(f"Execute {call['name']} -> {result}")

print()
print("Key observation: the model output two calls at once, and there is no dependency between them.")
print("The external program can execute them concurrently and feed both results back to the model at once.")


In [ ]:
# Sequential calls: the second call depends on the result of the first

user_query = "What is the temperature in Beijing today? What should I wear?"

# Round 1: the model first calls the weather tool
first_call = json.dumps({
    "tool_call": {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}
}, ensure_ascii=False)

print("=== Sequential call ===")
print(f"User: {user_query}")
print("\nRound 1 model output:")
print(first_call)

weather_result = execute_tool("get_weather", {"city": "Beijing", "unit": "celsius"})
print(f"\nTool result: {weather_result}")
print()

# Round 2: after seeing the weather result, the model decides to call a clothing-suggestion tool
wardrobe_tool_result = {"suggestion": "Long-sleeve shirt with a light jacket"}
print("Round 2 model output: based on weather 22C sunny, call the clothing-suggestion tool")
print(f"Tool result: {wardrobe_tool_result}")
print()
print("Key observation: the round-2 call depends on the round-1 result — the model must first know the temperature before it can suggest clothing.")
print("This dependency means a sequential call must span multiple rounds, with higher latency than parallel calls.")


## 4. Training a Model That Supports Function Call

A model is not born knowing how to output JSON-formatted tool calls. Faced with a request to "call the get_weather tool", an ordinary pretrained model outputs text that looks like JSON but is not strictly formatted, with arguments possibly fabricated. Making it reliably output a legal JSON call requires dedicated data construction and training.

The training data looks very similar to ordinary instruction-tuning data; the difference is that the assistant's reply contains a tool-call JSON. Let us look at a concrete training sample.


In [ ]:
# A complete Function Call training sample

training_sample = {
    "messages": [
        {
            "role": "system",
            "content": "You are an assistant that can use the following tools: get_weather(city) to query the weather; calculate(expression) to compute."
        },
        {
            "role": "user",
            "content": "What is the temperature in Beijing today?"
        },
        {
            "role": "assistant",
            "content": "",
            "tool_call": {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}
        },
        {
            "role": "tool",
            "name": "get_weather",
            "content": '{"temperature": 22, "condition": "sunny"}'
        },
        {
            "role": "assistant",
            "content": "Beijing is 22C and sunny today."
        }
    ]
}

print("=== A Function Call training sample ===")
print(json.dumps(training_sample, ensure_ascii=False, indent=2))
print()
print("This sample contains a full conversation flow: system -> user -> assistant(tool call) -> tool(result) -> assistant(final reply).")
print("Through many such samples, the model learns when to output a tool-call JSON.")


### 4.1 How the Loss Is Computed

The standard approach to training an LLM is next-token prediction. But in Function Call training, not every token should contribute to the loss. system messages, user messages, and tool result messages are all inputs to the model — the model should not be asked to "predict" them. What it should learn is what to output in the assistant position.

So during training a mask is used: tokens corresponding to the input parts (system + user + tool result) are marked 0 (do not count toward loss), and tokens corresponding to assistant outputs (both the tool-call JSON and the final reply) are marked 1 (count toward loss).


In [ ]:
# Loss mask demo: which tokens count toward the loss, which do not

# Render the training sample as continuous text and annotate whether each role's tokens count toward the loss
def render_sample(sample):
    # Render the multi-turn conversation as continuous text, annotating whether each role counts toward loss
    parts = []
    for msg in sample["messages"]:
        role = msg["role"]
        if role == "system":
            parts.append((f"[system] {msg['content']}\n", False))
        elif role == "user":
            parts.append((f"[user] {msg['content']}\n", False))
        elif role == "assistant" and msg.get("tool_call"):
            tc = msg["tool_call"]
            parts.append((f"[assistant-tool call] {json.dumps(tc, ensure_ascii=False)}\n", True))
        elif role == "assistant":
            parts.append((f"[assistant] {msg['content']}\n", True))
        elif role == "tool":
            parts.append((f"[tool-{msg['name']}] {msg['content']}\n", False))
    return parts

parts = render_sample(training_sample)

print("=== Loss Mask Demo ===")
print("(True = counts toward loss, False = does not)\n")
for text, mask in parts:
    label = "loss" if mask else "skip"
    short = text[:60].replace("\n", "") + ("..." if len(text) > 60 else "")
    print(f"[{label:6}] {short}")

print()
print("Key observation: only the assistant's outputs (tool-call JSON + final reply) count toward the loss.")
print("system/user/tool are all inputs; the model does not need to predict them.")


### 4.2 Where the Training Data Comes From

Constructing training samples is the most time-consuming part of Function Call training. There are three common data sources:

| Source | Approach | Pros and cons |
|:---|:---|:---|
| Human annotation | Annotators see the tool list and write "user question + expected tool call" pairs | High quality, but expensive and hard to scale |
| Strong-model distillation | A model such as GPT-4 that already supports tool calls generates call samples, which are then used as training data | Easy to scale, but capped by the strong model's ability |
| Real logs | Filter correct samples from existing tool-call logs of a product | Closest to the real distribution, but no logs exist during cold start |

In practice, projects usually mix all three: first use human annotation for a few thousand high-quality seed samples, then scale up to tens of thousands via strong-model distillation, and finally incorporate real logs for continuous optimization.


## 5. Error Handling and Failure Modes

A tool call does not always succeed on the first try. Real deployments encounter a variety of failures, and both the model and the external program must be able to handle them.

Below are the four most common failure modes, each paired with a concrete example.


In [ ]:
# Failure mode 1: tool execution failure
# For example, the weather API times out, or the user passes a city that does not exist

def execute_tool_with_failure(name, arguments):
    # A tool executor that may fail
    if name == "get_weather":
        city = arguments.get("city", "")
        # Simulate "city does not exist"
        if city not in ["Beijing", "Shanghai", "Guangzhou"]:
            return {"error": f"City not found: {city}"}
        return {"temperature": 22, "condition": "sunny"}
    return {"error": "Unknown tool"}

# Case: the model called a city that does not exist
result = execute_tool_with_failure("get_weather", {"city": "Atlantis"})
print("=== Failure mode 1: tool execution failure ===")
print("Model call: get_weather(city='Atlantis')")
print(f"Tool return: {result}")
print()
print("Engineering countermeasure: feed the error result back to the model as-is, so it generates an apology or a clarifying follow-up.")
print("Example: 'Sorry, weather information for Atlantis was not found. Which city did you mean?'")


In [ ]:
# Failure mode 2: the model picks the wrong tool
# The user asked for a computation, but the model called the weather tool

wrong_call = {"name": "get_weather", "arguments": {"city": "357*289"}}
print("=== Failure mode 2: the model picks the wrong tool ===")
print("User: what is 357 x 289?")
print(f"Model call: {wrong_call}")
print("Problem: the model treated the arithmetic expression as a city name")
print()
print("Engineering countermeasure: add more 'user-question type -> correct tool' samples to the training data.")
print("You can also emphasize each tool's applicable scenario in the system prompt.")


In [ ]:
# Failure mode 3: invalid arguments
# For example, the tool requires unit to be only celsius or fahrenheit, but the model passed kelvin

bad_call = {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "kelvin"}}
print("=== Failure mode 3: invalid arguments ===")
print(f"Model call: {bad_call}")
print("Problem: unit='kelvin' is not in enum ['celsius', 'fahrenheit']")
print()

# Engineering countermeasure: strict validation with a JSON Schema
valid_units = ["celsius", "fahrenheit"]
actual_unit = bad_call["arguments"]["unit"]
if actual_unit not in valid_units:
    print(f"Validation failed: unit must be one of {valid_units}, but received '{actual_unit}'")
    print("Engineering countermeasure: add schema validation in the external program; reject invalid arguments outright,")
    print("and feed the validation error back to the model so it can retry.")


In [ ]:
# Failure mode 4: tool-call infinite loop
# The model repeatedly calls the same tool and never outputs a final reply

print("=== Failure mode 4: tool-call infinite loop ===")
print("Scenario: the model calls get_weather several times in a row with identical arguments, never emitting a final reply")
print()

# Engineering countermeasure: set a maximum call count
MAX_TOOL_CALLS = 3
call_count = 0
for i in range(MAX_TOOL_CALLS + 2):  # simulate the model calling repeatedly
    call_count += 1
    if call_count > MAX_TOOL_CALLS:
        print(f"Call #{call_count}: exceeds the maximum {MAX_TOOL_CALLS}, force stop")
        break
    print(f"Call #{call_count}: get_weather(city='Beijing')")

print()
print("Engineering countermeasure: wrap the call loop with a counter; when the threshold is exceeded, terminate forcibly and return a fallback reply.")
print("Additionally, the training data can include positive samples that 'reply after one call' to reduce the model's tendency to call repeatedly.")


## 6. Modern Practice: From Function Call to Standardized Tool Calling

The name Function Call was first introduced by OpenAI in June 2023, when the API parameter was still called `functions`. Half a year later, OpenAI renamed it Tools, the parameter became `tools`, and the original `function_call` field was renamed to `tool_calls`. Behind this renaming is a more general design: a tool does not have to be a function; it can also be a retriever, code interpreter, file system, or anything else that can "be called". That is why the community now more commonly says Tool Use or Tool Calling, and Function Call has gradually become a historical name.

But each vendor's Tool Use interface still goes its own way — OpenAI has one format, Anthropic another, Google yet another. To support a single tool across multiple platforms, developers have to adapt repeatedly. At the end of 2024 Anthropic released MCP (Model Context Protocol), attempting to unify this ecosystem. The core idea of MCP is to standardize the description and call protocol of tools so that any MCP-compatible tool can be called directly by any MCP-supporting model, without separate adaptation per model.

The next step in this evolution is the Agent. When the model can call tools reliably, the external program can feed results back, and the model can decide the next step based on those results — these three things combined form a loop of "model-driven autonomous decisions". This loop is called the agent loop. From pure Function Call to agents, the essence is the recursive application of the same mechanism: call -> result -> decision -> call again.


## Summary

What this section covered:

- LLMs have two fundamental limitations: knowledge cutoff (cannot know events after training) and numerical computation errors (not good at precise arithmetic)
- Function Call separates "decision" from "execution": the model generates a structured JSON call request, the external program executes it
- The complete six-step flow: define tools -> inject into system prompt -> model outputs JSON -> external execution -> result feedback -> model generates final reply
- Multi-tool scenarios split into two kinds: parallel calls (no dependency, multiple can be generated at once) and sequential calls (with dependency, must span multiple rounds)
- Training a Function Call model needs dedicated samples: a complete conversation flow (system + user + assistant tool call + tool result + assistant final reply); the loss mask only counts the assistant's outputs
- Three data sources: human annotation, strong-model distillation, real logs
- Four common failure modes: tool execution failure, model picking the wrong tool, invalid arguments, infinite call loop
- Naming evolution: Function Call -> Tool Use -> MCP; behind it is the expansion from "calling functions" to "calling arbitrary tools" to "a standardized tool interface"


## Exercises

Three exercises cover three categories of content:

1. **Core mechanism**: hand-write a tool call JSON to understand the structure of the model's output
2. **Training data**: annotate the loss mask of a training sample to understand which tokens count toward loss
3. **Multi-tool calling**: judge whether a scenario needs parallel or sequential calls to understand dependencies

> You can ask an AI to help explain the approach, but it is not recommended to have the AI complete the exercise directly for you.


### Exercise 1: Hand-write a tool call JSON

Given a user question and a tool list, write the call JSON that the model should output.

Tool list:

- `calculate(expression)`: perform a mathematical computation
- `search(query)`: web search

User question: `What is 123 + 456 x 7?`

**Hint**: the model has to decide which tool to call and what arguments to pass. expression should be a string containing a mathematical expression.


In [ ]:
# Exercise 1: hand-write a tool call JSON
import json

# TODO: replace the placeholder string below with the JSON string you write
# Format: {"tool_call": {"name": "...", "arguments": {...}}}
your_answer = '''write your JSON string here'''

# Auto-check
assert your_answer != "write your JSON string here", "Please fill in your answer first"

try:
    parsed = json.loads(your_answer)
except json.JSONDecodeError as e:
    raise AssertionError(f"JSON format error: {e}")

assert "tool_call" in parsed, "The top level of the JSON should have a tool_call field"
assert parsed["tool_call"]["name"] == "calculate", "Should call the calculate tool"
assert "expression" in parsed["tool_call"]["arguments"], "arguments should contain an expression field"
# Check that the expression contains the required parts (operation order not strictly enforced)
expr = parsed["tool_call"]["arguments"]["expression"]
for token in ["123", "456", "7"]:
    assert token in expr, f"The expression should contain {token}"

print("Exercise 1 passed: you understand the JSON structure of a tool call")
print(f"Your answer: {parsed}")


### Exercise 2: Loss mask of a training sample

Below is the conversation flow of a training sample (simplified). For each message, decide whether the corresponding tokens should count toward the loss during training.

```text
[1] system:           "You can use the calculate(expression) tool"
[2] user:             "What is 12 x 8?"
[3] assistant(tool call): {"name": "calculate", "arguments": {"expression": "12*8"}}
[4] tool:             {"result": 96}
[5] assistant:        "12 x 8 equals 96."
```

Write the mask for each message as a list (True counts toward loss, False does not).

**Hint**: only positions the model needs to "generate" should count toward loss. Input positions (system/user/tool result) do not.


In [ ]:
# Exercise 2: loss mask of a training sample

# TODO: replace the placeholder list below with your answer
# Order corresponds to [1] system, [2] user, [3] assistant(tool call), [4] tool, [5] assistant
# Each element is True (counts toward loss) or False (does not)
your_mask = [None, None, None, None, None]

# Auto-check
assert your_mask != [None, None, None, None, None], "Please fill in your answer first"
assert len(your_mask) == 5, "There should be 5 elements"
assert all(isinstance(x, bool) for x in your_mask), "Each element should be True or False"

# Correct answer: system/user/tool are inputs (False); assistant is an output (True)
expected = [False, False, True, False, True]
assert your_mask == expected, "Incorrect answer. Hint: only the assistant's output positions count toward loss"

print("Exercise 2 passed: you understand the loss mask in Function Call training")
print(f"Your answer: {your_mask}")
print("Interpretation: [1]system [2]user are inputs; [3]assistant's tool call is content generated by the model;")
print("      [4]tool is the result returned by the external program; [5]assistant's final reply is also generated by the model.")


### Exercise 3: Parallel or sequential

For the three scenarios below, should each use parallel calls or sequential calls?

Scenario A: the user asks "What is the temperature in Beijing and in Shanghai today?"

Scenario B: the user asks "First check today's weather in Beijing, then recommend an outfit based on the temperature"

Scenario C: the user asks "What is 123 + 456? What is 789 - 123?"

Write the answer as a list where each element is the string `"parallel"` or `"sequential"`, in order A/B/C.

**Hint**: the criterion is whether later tool calls depend on earlier tool results. Dependency -> sequential; no dependency -> parallel.


In [ ]:
# Exercise 3: parallel or sequential

# TODO: replace the placeholder list below with your answer
# Order corresponds to scenarios A/B/C; each element is "parallel" or "sequential"
your_answer = [None, None, None]

# Auto-check
assert your_answer != [None, None, None], "Please fill in your answer first"
assert len(your_answer) == 3, "There should be 3 elements"
assert all(x in ["parallel", "sequential"] for x in your_answer), "Each element should be 'parallel' or 'sequential'"

# Correct answers:
# A: Beijing and Shanghai weather are independent of each other -> parallel
# B: outfit depends on the weather result -> sequential
# C: two independent computations -> parallel
expected = ["parallel", "sequential", "parallel"]

# Compare one by one and give feedback
for i, (y, e, scene) in enumerate(zip(your_answer, expected, ["A", "B", "C"])):
    if y != e:
        hint = {
            "A": "Hint: Beijing and Shanghai weather are independent, no dependency",
            "B": "Hint: the outfit recommendation depends on the weather result",
            "C": "Hint: two independent arithmetic problems, no dependency"
        }[scene]
        raise AssertionError(f"Scenario {scene} is wrong. {hint}")

print("Exercise 3 passed: you understand the criterion for parallel vs sequential")
print(f"Your answer: {your_answer}")
print("Interpretation:")
print("  A: two independent queries that can be called together -> parallel")
print("  B: the second call depends on the first result -> sequential")
print("  C: two independent computations -> parallel")
